# Embedding Visualization

Loads tree embeddings for the Air Passengers dataset at two embedding dimensions (`embedding_dim=1` and `embedding_dim=10`) and visualises them.

**Prerequisites:** both `local_hypertrees.ipynb` (for `embedding_dim=1`) and `embedding_ablation.ipynb` (for `embedding_dim=10`) must have been executed. The CSVs read are:

- `experiments/runs/results/local/airpassengers_tree_embeddings.csv` (dim=1, from local hypertrees)
- `experiments/runs/results/ablation/embedding_evaluation/airpassengers_10_tree_embeddings.csv` (dim=10, from embedding ablation)

**Output:** two PDF + two PNG files in `experiments/runs/results/plots/` — one pair per embedding dimension (`embedding_visualization_dim1_embeddings.{pdf,png}` and `embedding_visualization_dim10_embeddings.{pdf,png}`).

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import plotnine
from plotnine import *
from mizani.formatters import date_format

repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from experiments.utils import save_plot, load_embeddings_and_params

results_dir = repo_root / 'experiments' / 'runs' / 'results'
ablation_dir = results_dir / 'ablation' / 'embedding_evaluation'
plots_dir = results_dir / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)

plotnine.options.figure_size = (20, 10)

## Plot for `embedding_dim ∈ {1, 10}`

In [ ]:
color_map = {'Train': '#1f77b4', 'Test': '#FFA500'}
size_map  = {'Train': 1.3,       'Test': 1.3}

dataset = 'airpassengers'
for embedding_dim in [1, 10]:
    print(f'\n=== Loading CSVs for embedding_dim={embedding_dim} ===')
    embeds_long, _params_long, train_end = load_embeddings_and_params(
        dataset, embedding_dim, ablation_dir, results_dir,
    )

    embed_ncol = 1 if embedding_dim == 1 else 2
    embed_plot = (
        ggplot(embeds_long, aes(x='date', y='value', color='type'))
        + geom_line(aes(size='type'))
        + facet_wrap('embedding', scales='free_y', ncol=embed_ncol)
        + geom_vline(xintercept=train_end, linetype='dashed')
        + scale_color_manual(values=color_map)
        + scale_size_manual(values=size_map)
        + scale_x_datetime(labels=date_format('%Y'))
        + theme_bw(base_size=20)
        + theme(
            legend_position='none',
            legend_title=element_blank(),
            panel_grid_major_x=element_line(color='grey', alpha=0.05),
            panel_grid_minor_x=element_line(color='grey', alpha=0.05),
            panel_grid_major_y=element_line(color='grey', alpha=0.05),
            panel_grid_minor_y=element_line(color='grey', alpha=0.05),
        )
        + labs(title='', x='', y='')
    )
    save_plot(embed_plot, f'embedding_visualization_dim{embedding_dim}_embeddings', plots_dir)
    display(embed_plot)